# 🧠 NeuroVerse — Clock Drawing Test (CDT) Model Training

## Cognitive Module: Dementia Detection via Clock Drawing Analysis

### Clinical Background
The **Clock Drawing Test (CDT)** is one of the most widely used neuropsychological screening tools for dementia. 
Patients are asked to draw a clock showing "10 past 11" — this tests:
- **Visuospatial ability** (circle, number placement)
- **Executive function** (planning, sequencing)
- **Memory** (recall of clock face)
- **Attention** (following instructions)

### Shulman Scoring Scale (used in this dataset)
| Score | Meaning | AD Risk |
|-------|---------|---------|
| 5 | Perfect clock | Normal |
| 4 | Minor visuospatial errors | Borderline |
| 3 | Inaccurate representation (numbers present but wrong) | Mild impairment |
| 2 | Moderate disorganization | Moderate impairment |
| 1 | Severe disorganization | Severe impairment |
| 0 | No meaningful drawing | Severe impairment |

### Dataset
**DEMENTIA_DETECTION_CDT** from Roboflow (by Amisha)
- **7,365 images** of hand-drawn clocks
- **16 classes** (mapped to 6-class Shulman scale + binary AD/Normal)
- **97.3% validation accuracy** baseline

### Architecture
- **EfficientNet-B0** (pretrained on ImageNet) with custom classification head
- **Data augmentation** (rotation, perspective, blur — NO horizontal flip!)
- **GradCAM** for XAI saliency maps (shows what part of clock triggered prediction)

### Output
- `cognitive_cdt_model.pt` — PyTorch model for backend integration
- GradCAM visualization capability for XAI explanations

## 1️⃣ Environment Setup & Dependencies

In [ ]:
# ============================================================
# CELL 1: Install Dependencies (run once on Colab)
# ============================================================

!pip install -q torch torchvision timm albumentations grad-cam \
    matplotlib seaborn scikit-learn pandas numpy pillow tqdm roboflow \
    captum lime imbalanced-learn

import os
import sys
import json
import random
import warnings
import math
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, TensorDataset
import torchvision.transforms as T
from torchvision import models

import timm  # EfficientNet
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    roc_auc_score, f1_score, accuracy_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════
# Reproducibility
# ═══════════════════════════════════════════════════════════
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"   PyTorch: {torch.__version__}")
print(f"   timm: {timm.__version__}")

## 2️⃣ Dataset Download — Roboflow CDT Dataset

### Upload Methods:
| Method | How |
|--------|-----|
| **Roboflow API** (easiest) | Uses `roboflow` Python package — auto-downloads |
| **Google Drive** | Upload zip to Drive → mount → extract |
| **Direct Upload** | For small re-exports only |

### Steps:
1. Go to [Roboflow DEMENTIA_DETECTION_CDT](https://universe.roboflow.com/amisha/dementia_detection_cdt)
2. Click **"Use this Dataset"** → **Download** → Format: **Folder Structure** or **Classification**
3. Copy your **API key** from Roboflow settings
4. Paste it below and run!

In [ ]:
# ============================================================
# CELL 2: Dataset Download & Setup
# ============================================================

# ═══════════════════════════════════════════════════════════
# CONFIGURE: Choose your download method
# ═══════════════════════════════════════════════════════════
DOWNLOAD_METHOD = "roboflow"   # "roboflow", "drive", or "local"
ROBOFLOW_API_KEY = ""          # ← Paste your Roboflow API key here

# Google Drive settings (if using "drive" method)
DRIVE_FOLDER = "/content/drive/MyDrive/datasets"
CDT_ZIP_NAME = "DEMENTIA_DETECTION_CDT.zip"

# Paths
DATA_DIR = "./data/cognitive/cdt"
MODEL_DIR = "./models"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════
# METHOD 1: Roboflow API (recommended)
# ═══════════════════════════════════════════════════════════
if DOWNLOAD_METHOD == "roboflow":
    if not ROBOFLOW_API_KEY:
        print("⚠️  Please set your ROBOFLOW_API_KEY above!")
        print("   1. Go to https://app.roboflow.com/settings → API Key")
        print("   2. Paste it in the ROBOFLOW_API_KEY variable")
        print("   3. Re-run this cell")
    else:
        from roboflow import Roboflow
        rf = Roboflow(api_key=ROBOFLOW_API_KEY)
        project = rf.workspace("amisha").project("dementia_detection_cdt")
        version = project.version(2)  # Latest version
        dataset = version.download("folder", location=DATA_DIR)
        print(f"✅ Dataset downloaded to {DATA_DIR}")

# ═══════════════════════════════════════════════════════════
# METHOD 2: Google Drive
# ═══════════════════════════════════════════════════════════
elif DOWNLOAD_METHOD == "drive":
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        
        import zipfile, shutil
        zip_path = os.path.join(DRIVE_FOLDER, CDT_ZIP_NAME)
        
        if os.path.exists(zip_path):
            print(f"📂 Found: {CDT_ZIP_NAME}")
            print(f"   Extracting...")
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(DATA_DIR)
            file_count = sum(1 for _, _, fs in os.walk(DATA_DIR) for f in fs)
            print(f"   ✅ Extracted {file_count} files to {DATA_DIR}")
        else:
            print(f"⚠️  Not found: {zip_path}")
            print(f"   Upload '{CDT_ZIP_NAME}' to Google Drive → datasets/ folder")
    except ImportError:
        print("⚠️  Not running on Colab — use 'roboflow' or 'local' method")

# ═══════════════════════════════════════════════════════════
# METHOD 3: Local (files already extracted)
# ═══════════════════════════════════════════════════════════
elif DOWNLOAD_METHOD == "local":
    print(f"🖥️  Using local data from: {DATA_DIR}")
    if os.path.exists(DATA_DIR):
        file_count = sum(1 for _, _, fs in os.walk(DATA_DIR) for f in fs)
        print(f"   Found {file_count} files")
    else:
        print(f"   ❌ Directory not found! Extract dataset to {DATA_DIR}")

# ═══════════════════════════════════════════════════════════
# Scan the downloaded dataset
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📊 SCANNING DATASET STRUCTURE...")
print("=" * 60)

# Auto-detect folder structure (Roboflow exports as train/valid/test with class subfolders)
image_data = []
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if Path(f).suffix.lower() in IMAGE_EXTS:
            full_path = os.path.join(root, f)
            # Class = parent folder name
            class_name = os.path.basename(root)
            # Split = grandparent folder (train/valid/test)
            split_name = os.path.basename(os.path.dirname(root))
            image_data.append({
                'path': full_path,
                'class_raw': class_name,
                'split': split_name if split_name in ['train', 'valid', 'test'] else 'train'
            })

df_images = pd.DataFrame(image_data)
print(f"\n   Total images found: {len(df_images)}")
print(f"\n   Raw classes found:")
for cls, count in df_images['class_raw'].value_counts().items():
    print(f"     {cls}: {count}")
print(f"\n   Splits:")
for split, count in df_images['split'].value_counts().items():
    print(f"     {split}: {count}")

## 3️⃣ Class Mapping & Cleaning

The Roboflow dataset has 16 raw classes — some are duplicates. We map them to:
- **6-class Shulman scale** (0-5) for fine-grained scoring
- **Binary classification** (Normal vs Impaired) for AD screening

| Raw Classes | → Shulman Score | → Binary |
|------------|----------------|----------|
| `0`, `0_no_clock` | 0 | Impaired |
| `1`, `1_severe_vis` | 1 | Impaired |
| `2`, `2_mod_vis_xhands` | 2 | Impaired |
| `3`, `3_hands_vis_errors` | 3 | Impaired |
| `4`, `4_minor_VIS_errors` | 4 | Normal |
| `5`, `5_perfect_clock`, `Perfect`, `5_` | 5 | Normal |
| `score`, `Score`, `Unlabeled` | Excluded | Excluded |

In [ ]:
# ============================================================
# CELL 3: Class Mapping & Data Cleaning
# ============================================================

# ═══════════════════════════════════════════════════════════
# Map raw Roboflow classes → Shulman 0-5 scale
# ═══════════════════════════════════════════════════════════
CLASS_MAP = {
    # Score 0 — No meaningful drawing
    '0': 0, '0_no_clock': 0,
    # Score 1 — Severe disorganization
    '1': 1, '1_severe_vis': 1,
    # Score 2 — Moderate disorganization
    '2': 2, '2_mod_vis_xhands': 2,
    # Score 3 — Inaccurate representation
    '3': 3, '3_hands_vis_errors': 3,
    # Score 4 — Minor visuospatial errors
    '4': 4, '4_minor_VIS_errors': 4,
    # Score 5 — Perfect clock
    '5': 5, '5_': 5, '5_perfect_clock': 5, 'Perfect': 5,
}

# Classes to EXCLUDE (ambiguous/unlabeled)
EXCLUDE_CLASSES = {'score', 'Score', 'Unlabeled'}

# Binary mapping: Shulman ≥4 = Normal, ≤3 = Impaired
# (matches fusion_service.py: CDT_NORMAL = 4, CDT_IMPAIRED = 2)
BINARY_MAP = {0: 1, 1: 1, 2: 1, 3: 1, 4: 0, 5: 0}  # 0=Normal, 1=Impaired

# Class names
SHULMAN_NAMES = {
    0: '0-No Clock', 1: '1-Severe', 2: '2-Moderate', 
    3: '3-Inaccurate', 4: '4-Minor Errors', 5: '5-Perfect'
}
BINARY_NAMES = {0: 'Normal (4-5)', 1: 'Impaired (0-3)'}

# ═══════════════════════════════════════════════════════════
# Apply mapping
# ═══════════════════════════════════════════════════════════

# Remove excluded classes
before_count = len(df_images)
df_images = df_images[~df_images['class_raw'].isin(EXCLUDE_CLASSES)].copy()
excluded_count = before_count - len(df_images)

# Map to Shulman scores
df_images['shulman_score'] = df_images['class_raw'].map(CLASS_MAP)

# Remove any unmapped classes
unmapped = df_images[df_images['shulman_score'].isna()]
if len(unmapped) > 0:
    print(f"⚠️  Unmapped classes found: {unmapped['class_raw'].unique()}")
    df_images = df_images.dropna(subset=['shulman_score'])

df_images['shulman_score'] = df_images['shulman_score'].astype(int)

# Binary labels
df_images['binary_label'] = df_images['shulman_score'].map(BINARY_MAP)

# ═══════════════════════════════════════════════════════════
# Summary
# ═══════════════════════════════════════════════════════════
print("=" * 60)
print("📊 CLEANED DATASET SUMMARY")
print("=" * 60)
print(f"   Excluded: {excluded_count} images (unlabeled/ambiguous)")
print(f"   Remaining: {len(df_images)} images")

print(f"\n   📋 Shulman Scale Distribution:")
for score in sorted(df_images['shulman_score'].unique()):
    count = len(df_images[df_images['shulman_score'] == score])
    bar = '█' * (count // 50)
    print(f"     {SHULMAN_NAMES[score]:20s}: {count:5d} {bar}")

print(f"\n   📋 Binary Distribution:")
for label in sorted(df_images['binary_label'].unique()):
    count = len(df_images[df_images['binary_label'] == label])
    pct = count / len(df_images) * 100
    print(f"     {BINARY_NAMES[label]:20s}: {count:5d} ({pct:.1f}%)")

# Verify images are readable (sample check)
print(f"\n   🔍 Verifying random sample of images...")
bad_images = []
sample = df_images.sample(min(50, len(df_images)))
for _, row in sample.iterrows():
    try:
        img = Image.open(row['path'])
        img.verify()
    except Exception as e:
        bad_images.append(row['path'])

if bad_images:
    print(f"   ⚠️  {len(bad_images)} corrupt images found in sample — removing all corrupt")
    # Full scan
    valid_paths = []
    for _, row in tqdm(df_images.iterrows(), total=len(df_images), desc="   Validating"):
        try:
            img = Image.open(row['path'])
            img.verify()
            valid_paths.append(row['path'])
        except:
            pass
    df_images = df_images[df_images['path'].isin(valid_paths)]
    print(f"   ✅ {len(df_images)} valid images remaining")
else:
    print(f"   ✅ All sampled images are valid")

In [ ]:
# ============================================================
# CELL 4: Dataset Visualization — Sample Clock Drawings
# ============================================================

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
fig.suptitle('Sample Clock Drawings by Shulman Score', fontsize=16, fontweight='bold')

for score in range(6):
    subset = df_images[df_images['shulman_score'] == score]
    if len(subset) == 0:
        continue
    
    samples = subset.sample(min(2, len(subset)))
    for i, (_, row) in enumerate(samples.iterrows()):
        ax = axes[i, score]
        try:
            img = Image.open(row['path']).convert('RGB')
            ax.imshow(img)
        except:
            ax.text(0.5, 0.5, 'Error', ha='center', va='center', transform=ax.transAxes)
        
        ax.set_title(f"Score {score}: {SHULMAN_NAMES[score].split('-')[1]}", fontsize=9)
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'cdt_samples.png'), dpi=150, bbox_inches='tight')
plt.show()
print("← Score 0 (Severe)  ──────────────────────  Score 5 (Perfect) →")

## 4️⃣ Data Augmentation & DataLoaders

### Augmentation Strategy for Clock Drawings:
- **Rotation ±15°** — people tilt paper naturally
- **Perspective distortion** — simulates camera angle / scanner variation
- **Gaussian blur** — pen thickness & scan quality variation
- **Brightness/contrast jitter** — lighting conditions
- **Random affine shift** — centering variation
- **⚠️ NO horizontal flip** — counterclockwise numbers = dementia sign, not valid augmentation!

In [ ]:
# ============================================================
# CELL 5: Custom Dataset & Augmentation Pipeline
# ============================================================

# ═══════════════════════════════════════════════════════════
# Image size & normalization (EfficientNet-B0 expects 224x224)
# ═══════════════════════════════════════════════════════════
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ═══════════════════════════════════════════════════════════
# Augmentation transforms
# ═══════════════════════════════════════════════════════════
train_transforms = T.Compose([
    T.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),  # Resize slightly larger
    T.RandomCrop(IMG_SIZE),                     # Random crop to target size
    T.RandomRotation(15),                       # ±15° rotation (natural paper tilt)
    T.RandomPerspective(distortion_scale=0.1, p=0.3),  # Scanner angle
    T.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),                 # Slight position shift
        scale=(0.9, 1.1),                       # Minor scale variation
    ),
    T.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.1,
    ),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),  # Pen/scan blur
    T.RandomGrayscale(p=0.3),                   # Some are already grayscale
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    T.RandomErasing(p=0.1, scale=(0.02, 0.1)), # Small occlusions
])

val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ═══════════════════════════════════════════════════════════
# Custom Dataset
# ═══════════════════════════════════════════════════════════
class CDTDataset(Dataset):
    """Clock Drawing Test dataset for classification."""
    
    def __init__(self, dataframe, transform=None, task='shulman'):
        """
        Args:
            dataframe: DataFrame with 'path', 'shulman_score', 'binary_label'
            transform: Image transforms
            task: 'shulman' (6-class) or 'binary' (Normal/Impaired)
        """
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform
        self.task = task
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Load image
        image = Image.open(row['path']).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        # Label
        if self.task == 'shulman':
            label = row['shulman_score']
        else:
            label = row['binary_label']
        
        return image, label

# ═══════════════════════════════════════════════════════════
# Train/Val/Test Split
# ═══════════════════════════════════════════════════════════

# Check if Roboflow already provided splits
if set(df_images['split'].unique()) >= {'train', 'valid'}:
    print("✅ Using Roboflow's pre-defined splits")
    df_train = df_images[df_images['split'] == 'train']
    df_val = df_images[df_images['split'] == 'valid']
    df_test = df_images[df_images['split'] == 'test'] if 'test' in df_images['split'].values else None
else:
    print("📊 Creating stratified 70/15/15 split...")
    df_train, df_temp = train_test_split(
        df_images, test_size=0.3, random_state=SEED, stratify=df_images['shulman_score']
    )
    df_val, df_test = train_test_split(
        df_temp, test_size=0.5, random_state=SEED, stratify=df_temp['shulman_score']
    )

# If no separate test set, use validation as test
if df_test is None or len(df_test) == 0:
    df_test = df_val

print(f"\n   Train: {len(df_train)} images")
print(f"   Val:   {len(df_val)} images")
print(f"   Test:  {len(df_test)} images")

# ═══════════════════════════════════════════════════════════
# Weighted Sampler (handle class imbalance)
# ═══════════════════════════════════════════════════════════
TASK = 'shulman'  # 'shulman' for 6-class, 'binary' for 2-class
NUM_CLASSES = 6 if TASK == 'shulman' else 2

label_col = 'shulman_score' if TASK == 'shulman' else 'binary_label'

# Compute class weights for balanced sampling
class_counts = df_train[label_col].value_counts().sort_index()
class_weights = 1.0 / class_counts.values
sample_weights = df_train[label_col].map(
    dict(zip(class_counts.index, class_weights))
).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print(f"\n   Task: {TASK} ({NUM_CLASSES} classes)")
print(f"   Class weights for balanced sampling:")
for cls_idx, (count, weight) in enumerate(zip(class_counts.values, class_weights)):
    name = SHULMAN_NAMES.get(cls_idx, f'Class {cls_idx}') if TASK == 'shulman' else BINARY_NAMES.get(cls_idx)
    print(f"     {name}: {count} samples → weight {weight:.4f}")

# ═══════════════════════════════════════════════════════════
# DataLoaders
# ═══════════════════════════════════════════════════════════
BATCH_SIZE = 32

train_dataset = CDTDataset(df_train, transform=train_transforms, task=TASK)
val_dataset = CDTDataset(df_val, transform=val_transforms, task=TASK)
test_dataset = CDTDataset(df_test, transform=val_transforms, task=TASK)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

# Quick visualization of augmented samples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Augmented Training Samples', fontsize=14, fontweight='bold')
for i, (img, label) in enumerate(train_dataset):
    if i >= 10:
        break
    ax = axes[i // 5, i % 5]
    # Denormalize for display
    img_show = img.permute(1, 2, 0).numpy()
    img_show = img_show * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_show = np.clip(img_show, 0, 1)
    ax.imshow(img_show)
    name = SHULMAN_NAMES[label] if TASK == 'shulman' else BINARY_NAMES[label]
    ax.set_title(name, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\n✅ DataLoaders ready!")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

## 🔬 DDPM — Diffusion-Based Synthetic Clock Drawing Generation (Optional)

### Why Use Diffusion for CDT?
Some Shulman scores (e.g., 0 and 1) are heavily underrepresented in the dataset. While weighted sampling helps, generating **synthetic clock drawings** via DDPM provides genuinely new training examples rather than just re-weighting existing ones.

### Implementation:
- **Architecture**: Lightweight U-Net with sinusoidal time embedding + class conditioning (per Shulman score)
- **Resolution**: 64×64 generation → bicubic upscale to 224×224
- **Schedule**: Cosine β noise schedule (500 timesteps, Nichol & Dhariwal 2021)
- **Target**: Generate synthetic images for underrepresented Shulman score classes
- **Skip**: Set `USE_DDPM = False` if your dataset is already well-balanced or if Colab time is limited

### References:
- Ho et al. (2020) — *"Denoising Diffusion Probabilistic Models"* NeurIPS
- Nichol & Dhariwal (2021) — *"Improved DDPM"* ICML

In [ ]:
# ============================================================
# CELL 5B: DDPM — Class-Conditional Diffusion for CDT Images
# ============================================================
# Generates synthetic clock drawings to balance rare Shulman scores.
# Set USE_DDPM = False to skip (saves ~15 min Colab time).
# ============================================================

USE_DDPM = True  # Toggle diffusion data generation

if USE_DDPM:
    # ═══════════════════════════════════════════════════════════
    # DDPM Config
    # ═══════════════════════════════════════════════════════════
    DDPM_IMG_SIZE = 64          # Generate at 64×64, upscale later
    DDPM_TIMESTEPS = 500        # Diffusion steps
    DDPM_EPOCHS = 100           # Training epochs (~15 min on T4)
    DDPM_BATCH_SIZE = 64
    DDPM_LR = 2e-4
    NUM_SHULMAN = 6             # 6 conditioning classes
    DDPM_GEN_PER_CLASS = 200    # How many synthetic per underrepresented class

    # ═══════════════════════════════════════════════════════════
    # Cosine β schedule (Nichol & Dhariwal 2021)
    # ═══════════════════════════════════════════════════════════
    def cosine_beta_schedule(timesteps, s=0.008):
        steps = timesteps + 1
        x = torch.linspace(0, timesteps, steps)
        alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return torch.clamp(betas, 0.0001, 0.9999)

    betas = cosine_beta_schedule(DDPM_TIMESTEPS)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

    def q_sample(x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
        sqrt_ac = sqrt_alphas_cumprod[t].view(-1, 1, 1, 1).to(x_start.device)
        sqrt_omac = sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1).to(x_start.device)
        return sqrt_ac * x_start + sqrt_omac * noise

    # ═══════════════════════════════════════════════════════════
    # Lightweight U-Net with class conditioning
    # ═══════════════════════════════════════════════════════════
    class SinusoidalPosEmb(nn.Module):
        def __init__(self, dim):
            super().__init__()
            self.dim = dim
        def forward(self, t):
            half = self.dim // 2
            emb = math.log(10000) / (half - 1)
            emb = torch.exp(torch.arange(half, device=t.device) * -emb)
            emb = t[:, None].float() * emb[None, :]
            return torch.cat([emb.sin(), emb.cos()], dim=-1)

    class ResBlock(nn.Module):
        def __init__(self, ch_in, ch_out, t_dim):
            super().__init__()
            self.conv1 = nn.Conv2d(ch_in, ch_out, 3, padding=1)
            self.conv2 = nn.Conv2d(ch_out, ch_out, 3, padding=1)
            self.bn1 = nn.GroupNorm(8, ch_out)
            self.bn2 = nn.GroupNorm(8, ch_out)
            self.t_proj = nn.Linear(t_dim, ch_out)
            self.skip = nn.Conv2d(ch_in, ch_out, 1) if ch_in != ch_out else nn.Identity()

        def forward(self, x, t_emb):
            h = F.silu(self.bn1(self.conv1(x)))
            h = h + self.t_proj(t_emb)[:, :, None, None]
            h = F.silu(self.bn2(self.conv2(h)))
            return h + self.skip(x)

    class LightweightUNet(nn.Module):
        def __init__(self, in_ch=3, base_ch=64, t_dim=128, n_classes=6):
            super().__init__()
            self.time_mlp = nn.Sequential(
                SinusoidalPosEmb(t_dim), nn.Linear(t_dim, t_dim * 2), nn.SiLU(), nn.Linear(t_dim * 2, t_dim)
            )
            self.class_emb = nn.Embedding(n_classes, t_dim)

            # Encoder
            self.enc1 = ResBlock(in_ch, base_ch, t_dim)
            self.enc2 = ResBlock(base_ch, base_ch * 2, t_dim)
            self.pool = nn.MaxPool2d(2)

            # Bottleneck
            self.bot = ResBlock(base_ch * 2, base_ch * 2, t_dim)

            # Decoder
            self.up = nn.Upsample(scale_factor=2, mode='nearest')
            self.dec2 = ResBlock(base_ch * 4, base_ch, t_dim)
            self.dec1 = ResBlock(base_ch * 2, base_ch, t_dim)
            self.out_conv = nn.Conv2d(base_ch, in_ch, 1)

        def forward(self, x, t, y):
            t_emb = self.time_mlp(t) + self.class_emb(y)
            e1 = self.enc1(x, t_emb)
            e2 = self.enc2(self.pool(e1), t_emb)
            b = self.bot(self.pool(e2), t_emb)
            d2 = self.dec2(torch.cat([self.up(b), e2], 1), t_emb)
            d1 = self.dec1(torch.cat([self.up(d2), e1], 1), t_emb)
            return self.out_conv(d1)

    # ═══════════════════════════════════════════════════════════
    # Prepare DDPM training data (small 64×64 versions)
    # ═══════════════════════════════════════════════════════════
    ddpm_transform = T.Compose([
        T.Resize((DDPM_IMG_SIZE, DDPM_IMG_SIZE)),
        T.ToTensor(),
        T.Normalize([0.5]*3, [0.5]*3),  # Scale to [-1, 1]
    ])

    class DDPMDataset(Dataset):
        def __init__(self, dataframe, transform):
            self.data = dataframe.reset_index(drop=True)
            self.transform = transform
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            row = self.data.iloc[idx]
            img = Image.open(row['path']).convert('RGB')
            return self.transform(img), row['shulman_score']

    ddpm_dataset = DDPMDataset(df_train, ddpm_transform)
    ddpm_loader = DataLoader(ddpm_dataset, batch_size=DDPM_BATCH_SIZE, shuffle=True,
                             num_workers=2, pin_memory=True, drop_last=True)

    # ═══════════════════════════════════════════════════════════
    # Train DDPM
    # ═══════════════════════════════════════════════════════════
    ddpm_model = LightweightUNet(in_ch=3, base_ch=64, t_dim=128, n_classes=NUM_SHULMAN).to(device)
    ddpm_optimizer = optim.AdamW(ddpm_model.parameters(), lr=DDPM_LR, weight_decay=1e-5)
    ddpm_scheduler = optim.lr_scheduler.CosineAnnealingLR(ddpm_optimizer, T_max=DDPM_EPOCHS, eta_min=1e-6)

    param_count = sum(p.numel() for p in ddpm_model.parameters())
    print(f"🧬 DDPM U-Net: {param_count / 1e6:.1f}M parameters")
    print(f"   Training on {len(ddpm_dataset)} images at {DDPM_IMG_SIZE}×{DDPM_IMG_SIZE}")
    print(f"   Epochs: {DDPM_EPOCHS} | Batch: {DDPM_BATCH_SIZE}")
    print()

    best_ddpm_loss = float('inf')
    for ep in range(DDPM_EPOCHS):
        ddpm_model.train()
        ep_loss = 0
        for imgs, labels in ddpm_loader:
            imgs = imgs.to(device)
            labels = labels.long().to(device)
            t = torch.randint(0, DDPM_TIMESTEPS, (imgs.size(0),), device=device)
            noise = torch.randn_like(imgs)
            x_noisy = q_sample(imgs, t, noise)
            pred_noise = ddpm_model(x_noisy, t, labels)
            loss = F.mse_loss(pred_noise, noise)
            ddpm_optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ddpm_model.parameters(), 1.0)
            ddpm_optimizer.step()
            ep_loss += loss.item()
        ddpm_scheduler.step()
        avg_loss = ep_loss / len(ddpm_loader)
        if avg_loss < best_ddpm_loss:
            best_ddpm_loss = avg_loss
            torch.save(ddpm_model.state_dict(), os.path.join(MODEL_DIR, 'ddpm_cdt_best.pt'))
        if (ep + 1) % 20 == 0 or ep == 0:
            print(f"   DDPM Epoch {ep+1:3d}/{DDPM_EPOCHS} │ Loss: {avg_loss:.5f}")

    # ═══════════════════════════════════════════════════════════
    # Generate synthetic images for minority classes
    # ═══════════════════════════════════════════════════════════
    ddpm_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'ddpm_cdt_best.pt'), map_location=device))
    ddpm_model.eval()

    @torch.no_grad()
    def p_sample_loop(model, shape, y, timesteps, betas, alphas, alphas_cumprod, device):
        x = torch.randn(shape, device=device)
        for i in reversed(range(timesteps)):
            t = torch.full((shape[0],), i, device=device, dtype=torch.long)
            pred_noise = model(x, t, y)
            alpha = alphas[i]
            alpha_bar = alphas_cumprod[i]
            beta = betas[i]
            if i > 0:
                noise = torch.randn_like(x)
            else:
                noise = 0
            x = (1 / alpha.sqrt()) * (x - (beta / (1 - alpha_bar).sqrt()) * pred_noise)
            if i > 0:
                x = x + beta.sqrt() * noise
        return x.clamp(-1, 1)

    # Determine which classes need augmentation
    train_counts = df_train['shulman_score'].value_counts().sort_index()
    max_count = train_counts.max()
    
    print(f"\n📊 Class distribution before DDPM:")
    synthetic_dir = os.path.join(DATA_DIR, 'ddpm_synthetic')
    os.makedirs(synthetic_dir, exist_ok=True)
    
    new_rows = []
    total_gen = 0
    
    for score in range(NUM_SHULMAN):
        current = train_counts.get(score, 0)
        # Generate enough to bring class up to ≈ median count
        target = min(max_count, int(np.median(train_counts.values)))
        n_gen = max(0, min(DDPM_GEN_PER_CLASS, target - current))
        print(f"   Score {score} ({SHULMAN_NAMES[score]}): {current} → +{n_gen} synthetic")
        
        if n_gen > 0:
            class_dir = os.path.join(synthetic_dir, str(score))
            os.makedirs(class_dir, exist_ok=True)
            
            batch = min(n_gen, 32)
            generated = 0
            while generated < n_gen:
                n = min(batch, n_gen - generated)
                y = torch.full((n,), score, device=device, dtype=torch.long)
                samples = p_sample_loop(
                    ddpm_model, (n, 3, DDPM_IMG_SIZE, DDPM_IMG_SIZE), y,
                    DDPM_TIMESTEPS, betas, alphas, alphas_cumprod, device
                )
                # Upscale to 224×224 and save
                for j in range(n):
                    img = samples[j].cpu()
                    img = (img * 0.5 + 0.5).clamp(0, 1)  # [-1,1] → [0,1]
                    img = T.functional.resize(img, [IMG_SIZE, IMG_SIZE],
                                              interpolation=T.InterpolationMode.BICUBIC)
                    img_pil = T.ToPILImage()(img)
                    save_path = os.path.join(class_dir, f'ddpm_{score}_{generated + j:04d}.png')
                    img_pil.save(save_path)
                    new_rows.append({
                        'path': save_path,
                        'class_raw': f'ddpm_{score}',
                        'split': 'train',
                        'shulman_score': score,
                        'binary_label': BINARY_MAP[score],
                    })
                generated += n
                total_gen += n
    
    # Add synthetic images to training DataFrame
    if new_rows:
        df_synthetic = pd.DataFrame(new_rows)
        df_train = pd.concat([df_train, df_synthetic], ignore_index=True)
        
        # Rebuild DataLoader with new data
        label_col = 'shulman_score' if TASK == 'shulman' else 'binary_label'
        class_counts = df_train[label_col].value_counts().sort_index()
        class_weights = 1.0 / class_counts.values
        sample_weights = df_train[label_col].map(
            dict(zip(class_counts.index, class_weights))
        ).values
        sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
        
        train_dataset = CDTDataset(df_train, transform=train_transforms, task=TASK)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                                  num_workers=2, pin_memory=True)
        
        print(f"\n✅ DDPM: Generated {total_gen} synthetic images")
        print(f"   Training set: {len(df_train)} images (real + synthetic)")
        print(f"\n📊 Class distribution after DDPM:")
        for score in range(NUM_SHULMAN):
            count = len(df_train[df_train['shulman_score'] == score])
            print(f"   Score {score} ({SHULMAN_NAMES[score]}): {count}")
    else:
        print(f"\n✅ No synthetic generation needed — classes already balanced")

else:
    print("⏭️  DDPM disabled (USE_DDPM = False). Using real data only.")

## 5️⃣ Model Architecture — EfficientNet-B0 + Custom Head

### Why EfficientNet-B0?
- **Compound scaling** — balances depth, width, and resolution efficiently
- **5.3M parameters** — small enough for mobile deployment
- **ImageNet pretrained** — strong transfer learning baseline
- **Feature maps** at layer 7 are perfect for GradCAM visualization

### Architecture:
```
EfficientNet-B0 (frozen bottom 5 layers, fine-tune top 3)
    ↓ 1280-d feature vector
Dropout(0.3) → Linear(1280, 512) → ReLU → BatchNorm
    ↓
Dropout(0.2) → Linear(512, 128) → ReLU → BatchNorm
    ↓
Linear(128, NUM_CLASSES)  ← 6 for Shulman OR 2 for binary
```

In [ ]:
# ============================================================
# CELL 6: Model Architecture — CDTNet (EfficientNet-B0 based)
# ============================================================

class CDTNet(nn.Module):
    """
    Clock Drawing Test classifier based on EfficientNet-B0.
    
    Outputs:
    - shulman_score: Shulman 0-5 scale (or binary Normal/Impaired)
    - ad_risk: Continuous AD risk score [0, 1]
    
    GradCAM compatible — target_layer = self.backbone.conv_head
    """
    
    def __init__(self, num_classes=6, pretrained=True, dropout=0.3):
        super().__init__()
        
        # ── EfficientNet-B0 backbone ──
        self.backbone = timm.create_model(
            'efficientnet_b0', 
            pretrained=pretrained,
            num_classes=0,       # Remove original classification head
            global_pool='avg'    # Global average pooling → 1280-d vector
        )
        
        # Freeze bottom layers (first 5 blocks) for transfer learning
        # EfficientNet-B0 has 7 blocks (0-6)
        # Freeze blocks 0-4, fine-tune blocks 5-6 + head
        for name, param in self.backbone.named_parameters():
            if 'blocks.5' not in name and 'blocks.6' not in name and 'conv_head' not in name and 'bn2' not in name:
                param.requires_grad = False
        
        # Count trainable params
        trainable = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        frozen = sum(p.numel() for p in self.backbone.parameters() if not p.requires_grad)
        
        # ── Classification head ──
        feature_dim = self.backbone.num_features  # 1280 for EfficientNet-B0
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout * 0.67),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(128),
            nn.Linear(128, num_classes)
        )
        
        # ── AD Risk head (regression) ──
        # Maps Shulman score prediction to continuous risk
        self.risk_head = nn.Sequential(
            nn.Linear(num_classes, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Output in [0, 1]
        )
        
        head_params = sum(p.numel() for p in self.classifier.parameters()) + \
                      sum(p.numel() for p in self.risk_head.parameters())
        
        print(f"🏗️  CDTNet Architecture:")
        print(f"   Backbone: EfficientNet-B0 ({frozen:,} frozen + {trainable:,} trainable)")
        print(f"   Head: {head_params:,} parameters")
        print(f"   Total trainable: {trainable + head_params:,}")
        print(f"   Output: {num_classes} classes + AD risk score")
    
    def forward(self, x):
        features = self.backbone(x)          # [B, 1280]
        logits = self.classifier(features)   # [B, num_classes]
        
        # AD risk from logits (soft prediction)
        with torch.no_grad():
            probs = torch.softmax(logits, dim=-1)
        risk = self.risk_head(probs)         # [B, 1]
        
        return logits, risk.squeeze(-1)
    
    def get_features(self, x):
        """Get backbone features for GradCAM."""
        return self.backbone(x)

# ═══════════════════════════════════════════════════════════
# Instantiate model
# ═══════════════════════════════════════════════════════════
model = CDTNet(num_classes=NUM_CLASSES, pretrained=True, dropout=0.3).to(device)

# Verify with dummy input
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
logits, risk = model(dummy)
print(f"\n✅ Forward pass test:")
print(f"   Input:  {dummy.shape}")
print(f"   Logits: {logits.shape} (Shulman score predictions)")
print(f"   Risk:   {risk.shape} (AD risk scores)")
print(f"   Risk values: {risk.detach().cpu().numpy()}")

## 6️⃣ Training Configuration

### Training Strategy:
- **Loss**: Label Smoothing Cross-Entropy (prevents overconfidence)
- **Optimizer**: AdamW with weight decay (better generalization)
- **Scheduler**: Cosine Annealing with Warm Restarts
- **Early Stopping**: Patience 12 epochs on val F1
- **Mixed Precision**: FP16 training on GPU for speed

In [ ]:
# ============================================================
# CELL 7: Training Loop
# ============================================================

# ═══════════════════════════════════════════════════════════
# Hyperparameters
# ═══════════════════════════════════════════════════════════
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3       # For head (backbone uses 10x smaller)
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
PATIENCE = 12

# ═══════════════════════════════════════════════════════════
# Loss, Optimizer, Scheduler
# ═══════════════════════════════════════════════════════════

# Label smoothing cross-entropy
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

# Different learning rates: backbone (frozen layers already 0) vs head
backbone_params = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
head_params = list(model.classifier.parameters()) + list(model.risk_head.parameters())

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': LEARNING_RATE / 10},   # Fine-tune slowly
    {'params': head_params, 'lr': LEARNING_RATE},             # Train head faster
], weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# Mixed precision
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

# ═══════════════════════════════════════════════════════════
# Training functions
# ═══════════════════════════════════════════════════════════

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0
    all_preds, all_labels = [], []
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        if scaler:
            with torch.amp.autocast('cuda'):
                logits, risk = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, risk = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        logits, risk = model(images)
        loss = criterion(logits, labels)
        
        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs)
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1, np.array(all_preds), np.array(all_labels), np.array(all_probs)


# ═══════════════════════════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════════════════════════
print("=" * 60)
print("🚀 STARTING TRAINING")
print("=" * 60)
print(f"   Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}")
print(f"   Task: {TASK} ({NUM_CLASSES} classes)")
print(f"   Early stopping patience: {PATIENCE}")
print()

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': [],
    'train_f1': [], 'val_f1': [],
    'lr': []
}

best_val_f1 = 0
patience_counter = 0
best_epoch = 0

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device
    )
    
    # Validate
    val_loss, val_acc, val_f1, _, _, _ = validate(
        model, val_loader, criterion, device
    )
    
    # Scheduler step
    scheduler.step()
    current_lr = optimizer.param_groups[1]['lr']  # Head LR
    
    # Log history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)
    
    # Print progress
    print(f"  Epoch {epoch+1:3d}/{NUM_EPOCHS} │ "
          f"Train: loss={train_loss:.4f} acc={train_acc:.3f} f1={train_f1:.3f} │ "
          f"Val: loss={val_loss:.4f} acc={val_acc:.3f} f1={val_f1:.3f} │ "
          f"LR: {current_lr:.2e}", end="")
    
    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch + 1
        patience_counter = 0
        
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'val_acc': val_acc,
            'num_classes': NUM_CLASSES,
            'task': TASK,
            'class_names': SHULMAN_NAMES if TASK == 'shulman' else BINARY_NAMES,
            'img_size': IMG_SIZE,
        }, os.path.join(MODEL_DIR, 'cdt_best.pt'))
        
        print(f" ★ BEST", end="")
    else:
        patience_counter += 1
    
    print()  # Newline
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\n⏹️  Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
        break

print(f"\n{'='*60}")
print(f"✅ TRAINING COMPLETE!")
print(f"   Best epoch: {best_epoch} with val F1 = {best_val_f1:.4f}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CELL 8: Training Curves
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', linewidth=2)
axes[1].plot(history['val_acc'], label='Validation', linewidth=2)
axes[1].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[1].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1 Score
axes[2].plot(history['train_f1'], label='Train', linewidth=2)
axes[2].plot(history['val_f1'], label='Validation', linewidth=2)
axes[2].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[2].set_title('Weighted F1 Score', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'cdt_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7️⃣ Evaluation on Test Set

Load the best checkpoint and evaluate with:
- Confusion matrix
- Per-class precision/recall/F1
- ROC-AUC (one-vs-rest)

In [ ]:
# ============================================================
# CELL 9: Test Set Evaluation
# ============================================================

# Load best model
checkpoint = torch.load(os.path.join(MODEL_DIR, 'cdt_best.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']} (val F1={checkpoint['val_f1']:.4f})")

# Evaluate on test set
test_loss, test_acc, test_f1, test_preds, test_labels, test_probs = validate(
    model, test_loader, criterion, device
)

print(f"\n{'='*60}")
print(f"📊 TEST SET RESULTS")
print(f"{'='*60}")
print(f"   Accuracy:    {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"   F1 (weighted): {test_f1:.4f}")
print(f"   Loss:        {test_loss:.4f}")

# Per-class report
class_names = [SHULMAN_NAMES[i] for i in range(NUM_CLASSES)] if TASK == 'shulman' else \
              [BINARY_NAMES[i] for i in range(NUM_CLASSES)]
print(f"\n📋 Classification Report:")
print(classification_report(test_labels, test_preds, target_names=class_names, digits=3))

# ROC-AUC (one-vs-rest)
try:
    if NUM_CLASSES == 2:
        auc = roc_auc_score(test_labels, test_probs[:, 1])
        print(f"   ROC-AUC: {auc:.4f}")
    else:
        auc = roc_auc_score(test_labels, test_probs, multi_class='ovr', average='weighted')
        print(f"   ROC-AUC (weighted OVR): {auc:.4f}")
except Exception as e:
    print(f"   ROC-AUC: Could not compute ({e})")

# ═══════════════════════════════════════════════════════════
# Confusion Matrix
# ═══════════════════════════════════════════════════════════
cm = confusion_matrix(test_labels, test_preds)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'cdt_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════
# Per-class accuracy bar chart
# ═══════════════════════════════════════════════════════════
per_class_acc = cm.diagonal() / cm.sum(axis=1)
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#FF6B6B', '#FF8E72', '#FFC94D', '#95E06C', '#4ECDC4', '#45B7D1']
bars = ax.bar(class_names, per_class_acc, color=colors[:NUM_CLASSES], edgecolor='black', linewidth=0.5)
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{acc:.1%}', ha='center', fontweight='bold', fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Accuracy')
ax.set_title('Per-Class Accuracy', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'cdt_per_class_accuracy.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8️⃣ GradCAM — Explainable AI Visualization

### What is GradCAM?
**Gradient-weighted Class Activation Mapping** highlights which regions of the clock drawing the model focused on to make its prediction.

This is the **most visually compelling XAI output** for your app:
- Red regions = high contribution to prediction
- Blue regions = low contribution
- Shows doctors exactly WHY the model flagged a drawing as impaired

For clock drawings, GradCAM typically highlights:
- **Misplaced numbers** (e.g., "3" at the wrong position)
- **Incorrect hands** (wrong time shown)
- **Circle distortion** (asymmetric or broken)
- **Number spacing** (clustered on one side = hemispatial neglect)

In [ ]:
# ============================================================
# CELL 10: GradCAM Implementation
# ============================================================

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

class CDTModelWrapper(nn.Module):
    """Wrapper that returns only logits (GradCAM needs single output)."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        logits, _ = self.model(x)
        return logits

wrapped_model = CDTModelWrapper(model)

# Target layer for GradCAM — last convolutional layer of EfficientNet-B0
target_layer = model.backbone.conv_head

# Initialize GradCAM
cam = GradCAM(model=wrapped_model, target_layers=[target_layer])

# ═══════════════════════════════════════════════════════════
# Generate GradCAM visualizations for sample images
# ═══════════════════════════════════════════════════════════

def generate_gradcam(image_path, true_label=None):
    """Generate GradCAM for a single image."""
    # Load and preprocess
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    
    # Get prediction
    model.eval()
    with torch.no_grad():
        logits, risk = model(img_tensor)
        pred_class = logits.argmax(dim=1).item()
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        risk_val = risk.item()
    
    # Generate GradCAM for predicted class
    targets = [ClassifierOutputTarget(pred_class)]
    grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]
    
    # Prepare original image for overlay
    img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_resized).astype(np.float32) / 255.0
    
    # Overlay
    cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    
    return {
        'original': img_np,
        'gradcam': cam_image,
        'heatmap': grayscale_cam,
        'pred_class': pred_class,
        'true_label': true_label,
        'probs': probs,
        'risk': risk_val
    }


# ═══════════════════════════════════════════════════════════
# Visualize GradCAM for each Shulman score
# ═══════════════════════════════════════════════════════════
print("🔍 Generating GradCAM visualizations...")

fig, axes = plt.subplots(3, 6, figsize=(24, 12))
fig.suptitle('GradCAM Visualization by Shulman Score\n'
             '(Red = High Attention | Blue = Low Attention)', 
             fontsize=16, fontweight='bold')

for score in range(6):
    subset = df_test[df_test['shulman_score'] == score] if len(df_test[df_test['shulman_score'] == score]) > 0 else \
             df_images[df_images['shulman_score'] == score]
    
    if len(subset) == 0:
        for row in range(3):
            axes[row, score].axis('off')
        continue
    
    sample = subset.sample(1).iloc[0]
    result = generate_gradcam(sample['path'], true_label=score)
    
    # Row 0: Original
    axes[0, score].imshow(result['original'])
    axes[0, score].set_title(f"Score {score}\n{SHULMAN_NAMES[score]}", fontsize=10, fontweight='bold')
    axes[0, score].axis('off')
    if score == 0:
        axes[0, score].set_ylabel('Original', fontsize=12, fontweight='bold')
    
    # Row 1: GradCAM overlay
    axes[1, score].imshow(result['gradcam'])
    pred_name = SHULMAN_NAMES[result['pred_class']]
    color = 'green' if result['pred_class'] == score else 'red'
    axes[1, score].set_title(f"Pred: {pred_name}\nRisk: {result['risk']:.2f}", 
                             fontsize=9, color=color)
    axes[1, score].axis('off')
    if score == 0:
        axes[1, score].set_ylabel('GradCAM', fontsize=12, fontweight='bold')
    
    # Row 2: Heatmap only
    axes[2, score].imshow(result['heatmap'], cmap='jet')
    probs_str = f"Conf: {result['probs'][result['pred_class']]:.1%}"
    axes[2, score].set_title(probs_str, fontsize=9)
    axes[2, score].axis('off')
    if score == 0:
        axes[2, score].set_ylabel('Heatmap', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'cdt_gradcam_by_score.png'), dpi=150, bbox_inches='tight')
plt.show()

print("✅ GradCAM visualizations saved!")

In [ ]:
# ============================================================
# CELL 11: GradCAM — Correct vs Misclassified Analysis
# ============================================================

# Find correct and misclassified examples
correct_mask = test_preds == test_labels
correct_indices = np.where(correct_mask)[0]
wrong_indices = np.where(~correct_mask)[0]

print(f"   Correct: {len(correct_indices)} | Misclassified: {len(wrong_indices)}")

if len(wrong_indices) > 0:
    fig, axes = plt.subplots(2, min(5, len(wrong_indices)), figsize=(20, 8))
    fig.suptitle('GradCAM on MISCLASSIFIED Samples\n(What confused the model?)', 
                 fontsize=14, fontweight='bold')
    
    n_show = min(5, len(wrong_indices))
    if n_show == 1:
        axes = axes.reshape(2, 1)
    
    for i, idx in enumerate(wrong_indices[:n_show]):
        row = df_test.iloc[idx]
        result = generate_gradcam(row['path'], true_label=int(test_labels[idx]))
        
        # Original
        axes[0, i].imshow(result['original'])
        axes[0, i].set_title(
            f"True: {SHULMAN_NAMES[int(test_labels[idx])]}\n"
            f"Pred: {SHULMAN_NAMES[int(test_preds[idx])]}",
            fontsize=9, color='red', fontweight='bold'
        )
        axes[0, i].axis('off')
        
        # GradCAM
        axes[1, i].imshow(result['gradcam'])
        axes[1, i].set_title(f"Risk: {result['risk']:.2f}", fontsize=9)
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'cdt_misclassified_gradcam.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("🎉 No misclassifications on test set!")

## 🔍 Multi-XAI — Additional Explainability Methods

GradCAM gives a coarse heatmap, but different XAI methods reveal different facets of model reasoning. We add **4 more methods** for comprehensive explanations:

| Method | Library | What It Shows |
|--------|---------|---------------|
| **GradCAM++** | pytorch-grad-cam | Better localization for multiple focus regions |
| **LIME** | lime | Superpixel-based local explanations (model-agnostic) |
| **Integrated Gradients** | captum | Pixel-level attribution via path integral from baseline |
| **Occlusion Sensitivity** | captum | Systematic sliding-window importance mapping |

These are especially useful for clock drawings where the model may attend to multiple regions (numbers, hands, circle edge).

In [ ]:
# ============================================================
# CELL 10B: GradCAM++ — Better Multi-Region Localization
# ============================================================

from pytorch_grad_cam import GradCAMPlusPlus

cam_pp = GradCAMPlusPlus(model=wrapped_model, target_layers=[target_layer])

def generate_gradcampp(image_path, true_label=None):
    """Generate GradCAM++ for a single image."""
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        logits, risk = model(img_tensor)
        pred_class = logits.argmax(dim=1).item()
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    targets = [ClassifierOutputTarget(pred_class)]
    grayscale_cam = cam_pp(input_tensor=img_tensor, targets=targets)[0]
    
    img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_resized).astype(np.float32) / 255.0
    cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    
    return img_np, cam_image, grayscale_cam, pred_class, probs

# Show GradCAM++ for one sample per Shulman score
fig, axes = plt.subplots(2, 6, figsize=(24, 8))
fig.suptitle('GradCAM++ — Better Multi-Region Localization', fontsize=16, fontweight='bold')

for score in range(6):
    subset = df_test[df_test['shulman_score'] == score] if len(df_test[df_test['shulman_score'] == score]) > 0 else \
             df_images[df_images['shulman_score'] == score]
    if len(subset) == 0:
        for r in range(2): axes[r, score].axis('off')
        continue
    sample = subset.sample(1).iloc[0]
    img_np, cam_img, heatmap, pred, probs = generate_gradcampp(sample['path'], score)
    
    axes[0, score].imshow(img_np)
    axes[0, score].set_title(f"Score {score}\n{SHULMAN_NAMES[score]}", fontsize=10, fontweight='bold')
    axes[0, score].axis('off')
    
    axes[1, score].imshow(cam_img)
    color = 'green' if pred == score else 'red'
    axes[1, score].set_title(f"Pred: {SHULMAN_NAMES[pred]}\nConf: {probs[pred]:.1%}", fontsize=9, color=color)
    axes[1, score].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'cdt_gradcampp.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ GradCAM++ visualizations saved!")

In [ ]:
# ============================================================
# CELL 10C: LIME — Superpixel-Based Local Explanations
# ============================================================

from lime import lime_image

lime_explainer = lime_image.LimeImageExplainer()

def lime_predict_fn(images_np):
    """LIME callback — takes numpy batch [N, H, W, 3], returns probs [N, C]."""
    batch = torch.stack([val_transforms(Image.fromarray(img.astype('uint8'))) for img in images_np])
    batch = batch.to(device)
    model.eval()
    with torch.no_grad():
        logits, _ = model(batch)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    return probs

# Generate LIME for representative samples
fig, axes = plt.subplots(2, 6, figsize=(24, 8))
fig.suptitle('LIME — Superpixel Explanations\n(Green = supports prediction | Red = contradicts)',
             fontsize=14, fontweight='bold')

for score in range(6):
    subset = df_test[df_test['shulman_score'] == score] if len(df_test[df_test['shulman_score'] == score]) > 0 else \
             df_images[df_images['shulman_score'] == score]
    if len(subset) == 0:
        for r in range(2): axes[r, score].axis('off')
        continue
    sample = subset.sample(1).iloc[0]
    
    img_pil = Image.open(sample['path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_pil)
    
    # Run LIME (fewer samples for speed)
    explanation = lime_explainer.explain_instance(
        img_np, lime_predict_fn,
        top_labels=1, hide_color=0, num_samples=500,
        batch_size=32
    )
    
    # Get predicted label
    model.eval()
    img_tensor = val_transforms(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(img_tensor)[0].argmax(dim=1).item()
    
    # Positive/negative mask
    temp_img, mask = explanation.get_image_and_mask(
        pred, positive_only=False, num_features=10,
        hide_rest=False, min_weight=0.01
    )
    
    axes[0, score].imshow(img_np)
    axes[0, score].set_title(f"Score {score}: {SHULMAN_NAMES[score]}", fontsize=10, fontweight='bold')
    axes[0, score].axis('off')
    
    axes[1, score].imshow(temp_img)
    color = 'green' if pred == score else 'red'
    axes[1, score].set_title(f"Pred: {SHULMAN_NAMES[pred]}", fontsize=9, color=color)
    axes[1, score].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'cdt_lime.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ LIME explanations saved!")

In [ ]:
# ============================================================
# CELL 10D: Integrated Gradients + Occlusion Sensitivity
# ============================================================

from captum.attr import IntegratedGradients, Occlusion

# ── Integrated Gradients ──
ig = IntegratedGradients(wrapped_model)

def compute_ig(image_path):
    """Compute Integrated Gradients pixel-level attributions."""
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    img_tensor.requires_grad = True
    
    model.eval()
    with torch.no_grad():
        pred_class = wrapped_model(img_tensor).argmax(dim=1).item()
    
    # Baseline = black image after normalization
    baseline = torch.zeros_like(img_tensor).to(device)
    
    attr = ig.attribute(img_tensor, baselines=baseline, target=pred_class,
                        n_steps=50, internal_batch_size=32)
    # Aggregate across channels
    attr_np = attr.squeeze().cpu().detach().numpy()
    attr_np = np.mean(np.abs(attr_np), axis=0)  # [H, W]
    attr_np = (attr_np - attr_np.min()) / (attr_np.max() - attr_np.min() + 1e-8)
    return attr_np, pred_class

# ── Occlusion Sensitivity ──
occ = Occlusion(wrapped_model)

def compute_occlusion(image_path):
    """Compute Occlusion Sensitivity map."""
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        pred_class = wrapped_model(img_tensor).argmax(dim=1).item()
    
    attr = occ.attribute(img_tensor, target=pred_class,
                         strides=(3, 8, 8),
                         sliding_window_shapes=(3, 15, 15),
                         baselines=0)
    attr_np = attr.squeeze().cpu().detach().numpy()
    attr_np = np.mean(np.abs(attr_np), axis=0)
    attr_np = (attr_np - attr_np.min()) / (attr_np.max() - attr_np.min() + 1e-8)
    return attr_np, pred_class

# ═══════════════════════════════════════════════════════════
# Compare all 5 XAI methods side-by-side
# ═══════════════════════════════════════════════════════════

print("🔍 Generating Multi-XAI comparison (this may take a few minutes)...")

# Pick 3 representative samples: perfect (5), mild (3), severe (1)
representative_scores = [5, 3, 1]
fig, axes = plt.subplots(len(representative_scores), 6, figsize=(30, 5 * len(representative_scores)))
fig.suptitle('Multi-XAI Comparison — 5 Explanation Methods',
             fontsize=18, fontweight='bold', y=1.02)

col_titles = ['Original', 'GradCAM', 'GradCAM++', 'LIME', 'Integrated\nGradients', 'Occlusion\nSensitivity']

for row_idx, score in enumerate(representative_scores):
    subset = df_test[df_test['shulman_score'] == score] if len(df_test[df_test['shulman_score'] == score]) > 0 else \
             df_images[df_images['shulman_score'] == score]
    if len(subset) == 0:
        for c in range(6): axes[row_idx, c].axis('off')
        continue
    
    sample = subset.sample(1).iloc[0]
    img_path = sample['path']
    img_pil = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_pil).astype(np.float32) / 255.0
    img_tensor = val_transforms(Image.open(img_path).convert('RGB')).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        pred = wrapped_model(img_tensor).argmax(dim=1).item()
    
    # Col 0: Original
    axes[row_idx, 0].imshow(img_np)
    axes[row_idx, 0].set_ylabel(f"Score {score}\n{SHULMAN_NAMES[score]}", fontsize=12, fontweight='bold')
    if row_idx == 0: axes[row_idx, 0].set_title(col_titles[0], fontsize=12, fontweight='bold')
    axes[row_idx, 0].axis('off')
    
    # Col 1: GradCAM
    gc = cam(input_tensor=img_tensor, targets=[ClassifierOutputTarget(pred)])[0]
    axes[row_idx, 1].imshow(show_cam_on_image(img_np, gc, use_rgb=True))
    if row_idx == 0: axes[row_idx, 1].set_title(col_titles[1], fontsize=12, fontweight='bold')
    axes[row_idx, 1].axis('off')
    
    # Col 2: GradCAM++
    gcpp = cam_pp(input_tensor=img_tensor, targets=[ClassifierOutputTarget(pred)])[0]
    axes[row_idx, 2].imshow(show_cam_on_image(img_np, gcpp, use_rgb=True))
    if row_idx == 0: axes[row_idx, 2].set_title(col_titles[2], fontsize=12, fontweight='bold')
    axes[row_idx, 2].axis('off')
    
    # Col 3: LIME
    img_uint8 = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE)))
    explanation = lime_explainer.explain_instance(img_uint8, lime_predict_fn,
                                                  top_labels=1, hide_color=0,
                                                  num_samples=300, batch_size=32)
    temp_img, _ = explanation.get_image_and_mask(pred, positive_only=False,
                                                  num_features=10, hide_rest=False)
    axes[row_idx, 3].imshow(temp_img)
    if row_idx == 0: axes[row_idx, 3].set_title(col_titles[3], fontsize=12, fontweight='bold')
    axes[row_idx, 3].axis('off')
    
    # Col 4: Integrated Gradients
    ig_map, _ = compute_ig(img_path)
    axes[row_idx, 4].imshow(img_np)
    axes[row_idx, 4].imshow(ig_map, cmap='hot', alpha=0.6)
    if row_idx == 0: axes[row_idx, 4].set_title(col_titles[4], fontsize=12, fontweight='bold')
    axes[row_idx, 4].axis('off')
    
    # Col 5: Occlusion
    occ_map, _ = compute_occlusion(img_path)
    axes[row_idx, 5].imshow(img_np)
    axes[row_idx, 5].imshow(occ_map, cmap='hot', alpha=0.6)
    if row_idx == 0: axes[row_idx, 5].set_title(col_titles[5], fontsize=12, fontweight='bold')
    axes[row_idx, 5].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'cdt_multi_xai_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print("✅ Multi-XAI comparison saved!")
print("   Methods: GradCAM | GradCAM++ | LIME | Integrated Gradients | Occlusion Sensitivity")

## 9️⃣ Unfreeze & Fine-tune Full Model (Optional)

After the head is well-trained, optionally unfreeze ALL backbone layers for a few more epochs at a very low learning rate. This can improve accuracy by 1-3%.

In [ ]:
# ============================================================
# CELL 12: Full Fine-tuning (Optional — run for extra 1-3% accuracy)
# ============================================================

FULL_FINETUNE = True  # Set False to skip

if FULL_FINETUNE:
    print("🔓 Unfreezing ALL backbone layers for full fine-tuning...")
    
    # Unfreeze everything
    for param in model.parameters():
        param.requires_grad = True
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Trainable parameters: {total_params:,}")
    
    # Very low learning rate for full fine-tuning
    FT_LR = 1e-5
    FT_EPOCHS = 15
    FT_PATIENCE = 8
    
    ft_optimizer = optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-5)
    ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=FT_EPOCHS, eta_min=1e-7)
    
    print(f"   LR: {FT_LR} | Epochs: {FT_EPOCHS} | Patience: {FT_PATIENCE}")
    print()
    
    ft_patience_counter = 0
    
    for epoch in range(FT_EPOCHS):
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, criterion, ft_optimizer, scaler, device
        )
        val_loss, val_acc, val_f1, _, _, _ = validate(model, val_loader, criterion, device)
        ft_scheduler.step()
        
        print(f"  FT Epoch {epoch+1:2d}/{FT_EPOCHS} │ "
              f"Train: f1={train_f1:.3f} │ Val: f1={val_f1:.3f}", end="")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            ft_patience_counter = 0
            torch.save({
                'epoch': NUM_EPOCHS + epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': ft_optimizer.state_dict(),
                'val_f1': val_f1,
                'val_acc': val_acc,
                'num_classes': NUM_CLASSES,
                'task': TASK,
                'class_names': SHULMAN_NAMES if TASK == 'shulman' else BINARY_NAMES,
                'img_size': IMG_SIZE,
                'full_finetuned': True,
            }, os.path.join(MODEL_DIR, 'cdt_best.pt'))
            print(f" ★ NEW BEST (f1={val_f1:.4f})")
        else:
            ft_patience_counter += 1
            print()
        
        if ft_patience_counter >= FT_PATIENCE:
            print(f"\n⏹️  Fine-tuning stopped (no improvement for {FT_PATIENCE} epochs)")
            break
    
    print(f"\n✅ Fine-tuning complete! Best F1: {best_val_f1:.4f}")
    
    # Reload best
    checkpoint = torch.load(os.path.join(MODEL_DIR, 'cdt_best.pt'), map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    print("⏭️  Skipping full fine-tuning")

## 🔟 Model Export for NeuroVerse Backend

Export the trained model in formats ready for deployment:
1. **`cognitive_cdt_model.pt`** — Full PyTorch model for backend (`app/ml/models/`)
2. **`cognitive_cdt_model_mobile.pt`** — TorchScript for mobile (optional)
3. **`cdt_class_config.json`** — Class mapping + thresholds for `fusion_service.py`
4. **GradCAM inference function** — Ready to paste into `xai_service.py`

In [ ]:
# ============================================================
# CELL 13: Model Export
# ============================================================

EXPORT_DIR = os.path.join(MODEL_DIR, "export")
os.makedirs(EXPORT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════
# 1. Export full PyTorch model (for backend)
# ═══════════════════════════════════════════════════════════
export_path = os.path.join(EXPORT_DIR, "cognitive_cdt_model.pt")

torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'architecture': 'efficientnet_b0',
        'num_classes': NUM_CLASSES,
        'task': TASK,
        'img_size': IMG_SIZE,
        'dropout': 0.3,
    },
    'class_mapping': {
        'shulman_names': SHULMAN_NAMES,
        'binary_names': BINARY_NAMES,
        'binary_map': BINARY_MAP,
        'class_map': CLASS_MAP,
    },
    'normalization': {
        'mean': IMAGENET_MEAN,
        'std': IMAGENET_STD,
    },
    'metrics': {
        'test_accuracy': test_acc,
        'test_f1': test_f1,
        'best_val_f1': best_val_f1,
    },
    'clinical_thresholds': {
        'cdt_max': 5,
        'cdt_normal': 4,       # ≥4 = normal (matches fusion_service.py)
        'cdt_impaired': 2,     # ≤2 = significant impairment
        'ad_risk_mapping': {   # Shulman score → AD risk range
            5: [0.02, 0.10],   # Perfect clock → very low risk
            4: [0.08, 0.20],   # Minor errors → low risk  
            3: [0.20, 0.45],   # Inaccurate → moderate risk
            2: [0.40, 0.70],   # Moderate disorg → high risk
            1: [0.60, 0.90],   # Severe disorg → very high risk
            0: [0.80, 0.98],   # No clock → critical risk
        }
    }
}, export_path)

size_mb = os.path.getsize(export_path) / (1024 * 1024)
print(f"✅ Full model exported: {export_path} ({size_mb:.1f} MB)")

# ═══════════════════════════════════════════════════════════
# 2. TorchScript model (for mobile / ONNX conversion)
# ═══════════════════════════════════════════════════════════
try:
    model.eval()
    # Create a wrapper that only returns logits for TorchScript
    class CDTInference(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.backbone = model.backbone
            self.classifier = model.classifier
        
        def forward(self, x):
            features = self.backbone(x)
            logits = self.classifier(features)
            return logits
    
    inference_model = CDTInference(model)
    scripted = torch.jit.trace(inference_model, torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device))
    script_path = os.path.join(EXPORT_DIR, "cognitive_cdt_model_mobile.pt")
    scripted.save(script_path)
    size_mb_s = os.path.getsize(script_path) / (1024 * 1024)
    print(f"✅ TorchScript model: {script_path} ({size_mb_s:.1f} MB)")
except Exception as e:
    print(f"⚠️  TorchScript export failed (non-critical): {e}")

# ═══════════════════════════════════════════════════════════
# 3. Class config JSON (for backend integration)
# ═══════════════════════════════════════════════════════════
config = {
    "model_name": "CDTNet",
    "version": "1.0",
    "architecture": "efficientnet_b0",
    "task": TASK,
    "num_classes": NUM_CLASSES,
    "img_size": IMG_SIZE,
    "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
    "class_names": {str(k): v for k, v in SHULMAN_NAMES.items()},
    "shulman_to_risk": {
        "5": {"ad_risk": 0.05, "pd_risk": 0.02, "label": "Normal"},
        "4": {"ad_risk": 0.15, "pd_risk": 0.05, "label": "Borderline"},
        "3": {"ad_risk": 0.35, "pd_risk": 0.08, "label": "Mild Impairment"},
        "2": {"ad_risk": 0.55, "pd_risk": 0.10, "label": "Moderate Impairment"},
        "1": {"ad_risk": 0.75, "pd_risk": 0.12, "label": "Severe Impairment"},
        "0": {"ad_risk": 0.90, "pd_risk": 0.15, "label": "No Clock / Critical"},
    },
    "metrics": {
        "test_accuracy": round(test_acc, 4),
        "test_f1": round(test_f1, 4),
        "best_val_f1": round(best_val_f1, 4),
    }
}

config_path = os.path.join(EXPORT_DIR, "cdt_class_config.json")
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Config JSON: {config_path}")

# ═══════════════════════════════════════════════════════════
# Summary
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"📦 EXPORTED FILES:")
print(f"{'='*60}")
for f in os.listdir(EXPORT_DIR):
    fpath = os.path.join(EXPORT_DIR, f)
    fsize = os.path.getsize(fpath) / (1024 * 1024)
    print(f"   {f:40s} {fsize:8.2f} MB")
print(f"\n   Copy these to your backend:")
print(f"   cognitive_cdt_model.pt  → neuroverse-backend/app/ml/models/")
print(f"   cdt_class_config.json   → neuroverse-backend/app/ml/config/")

## 1️⃣1️⃣ Backend Integration Code

Ready-to-use Python code for your NeuroVerse backend. This shows how to:
1. Load the model in FastAPI
2. Predict Shulman score from an uploaded clock drawing image
3. Generate GradCAM heatmap for XAI

In [ ]:
# ============================================================
# CELL 14: Backend Integration Code (copy to your backend)
# ============================================================

backend_code = '''
# ═══════════════════════════════════════════════════════════
# FILE: neuroverse-backend/app/ml/predictors/cognitive_predictor.py
# ═══════════════════════════════════════════════════════════
"""
CDT (Clock Drawing Test) Predictor for NeuroVerse Backend.
Loads the trained EfficientNet-B0 model and predicts Shulman score + AD risk.
"""
import os
import json
import torch
import torch.nn as nn
import timm
import numpy as np
from PIL import Image
from torchvision import transforms

# ── Model Architecture (must match training) ──
class CDTNet(nn.Module):
    def __init__(self, num_classes=6, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0', pretrained=False, num_classes=0, global_pool='avg'
        )
        feature_dim = self.backbone.num_features  # 1280
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim, 512), nn.ReLU(inplace=True), nn.BatchNorm1d(512),
            nn.Dropout(dropout * 0.67),
            nn.Linear(512, 128), nn.ReLU(inplace=True), nn.BatchNorm1d(128),
            nn.Linear(128, num_classes)
        )
        self.risk_head = nn.Sequential(
            nn.Linear(num_classes, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        with torch.no_grad():
            probs = torch.softmax(logits, dim=-1)
        risk = self.risk_head(probs)
        return logits, risk.squeeze(-1)


class CDTPredictor:
    """Clock Drawing Test predictor for the NeuroVerse backend."""
    
    SHULMAN_TO_RISK = {
        5: {"ad_risk": 0.05, "pd_risk": 0.02, "label": "Normal"},
        4: {"ad_risk": 0.15, "pd_risk": 0.05, "label": "Borderline"},
        3: {"ad_risk": 0.35, "pd_risk": 0.08, "label": "Mild Impairment"},
        2: {"ad_risk": 0.55, "pd_risk": 0.10, "label": "Moderate Impairment"},
        1: {"ad_risk": 0.75, "pd_risk": 0.12, "label": "Severe Impairment"},
        0: {"ad_risk": 0.90, "pd_risk": 0.15, "label": "No Clock / Critical"},
    }
    
    def __init__(self, model_path: str = None):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        if model_path is None:
            model_path = os.path.join(
                os.path.dirname(__file__), '..', 'models', 'cognitive_cdt_model.pt'
            )
        
        # Load checkpoint
        checkpoint = torch.load(model_path, map_location=self.device)
        config = checkpoint['model_config']
        
        # Build model
        self.model = CDTNet(
            num_classes=config['num_classes'],
            dropout=config['dropout']
        ).to(self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        
        # Preprocessing
        self.img_size = config['img_size']
        norm = checkpoint['normalization']
        self.transform = transforms.Compose([
            transforms.Resize((self.img_size, self.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=norm['mean'], std=norm['std']),
        ])
        
        self.class_mapping = checkpoint['class_mapping']
        print(f"✅ CDT model loaded ({config['num_classes']} classes, {self.device})")
    
    def predict(self, image_path: str) -> dict:
        """Predict Shulman score and AD risk from a clock drawing image."""
        img = Image.open(image_path).convert('RGB')
        tensor = self.transform(img).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            logits, risk = self.model(tensor)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            pred_score = int(logits.argmax(dim=1).item())
            confidence = float(probs[pred_score])
        
        risk_info = self.SHULMAN_TO_RISK[pred_score]
        
        return {
            "shulman_score": pred_score,
            "confidence": confidence,
            "class_probabilities": {str(i): float(p) for i, p in enumerate(probs)},
            "ad_risk": risk_info["ad_risk"],
            "pd_risk": risk_info["pd_risk"],
            "clinical_label": risk_info["label"],
            "is_normal": pred_score >= 4,
            "is_impaired": pred_score <= 2,
        }
    
    def predict_from_bytes(self, image_bytes: bytes) -> dict:
        """Predict from raw image bytes (for API endpoint)."""
        import io
        img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        # Save temporarily and predict
        import tempfile
        with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
            img.save(f.name)
            result = self.predict(f.name)
            os.unlink(f.name)
        return result


# ═══════════════════════════════════════════════════════════
# Usage in FastAPI endpoint:
# ═══════════════════════════════════════════════════════════
#
# predictor = CDTPredictor()
#
# @router.post("/predict/clock-drawing")
# async def predict_clock_drawing(file: UploadFile):
#     image_bytes = await file.read()
#     result = predictor.predict_from_bytes(image_bytes)
#     return result
'''

print(backend_code)
print("\n" + "=" * 60)
print("📋 Copy the code above to:")
print("   neuroverse-backend/app/ml/predictors/cognitive_predictor.py")
print("=" * 60)

## 1️⃣2️⃣ Inference Demo — Test on New Clock Drawings

In [ ]:
# ============================================================
# CELL 15: Inference Demo — Upload & Predict on New Images
# ============================================================

def predict_and_visualize(image_path, show_gradcam=True):
    """Full inference pipeline with GradCAM visualization."""
    
    # Load image
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        logits, risk = model(img_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_score = logits.argmax(dim=1).item()
        risk_val = risk.item()
    
    # Clinical interpretation
    shulman_to_risk = {
        5: ("Normal", "Low"), 4: ("Borderline", "Low-Moderate"),
        3: ("Mild Impairment", "Moderate"), 2: ("Moderate Impairment", "High"),
        1: ("Severe Impairment", "Very High"), 0: ("Critical", "Critical")
    }
    clinical_label, risk_level = shulman_to_risk[pred_score]
    
    if show_gradcam:
        # GradCAM
        targets = [ClassifierOutputTarget(pred_score)]
        grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]
        img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
        img_np = np.array(img_resized).astype(np.float32) / 255.0
        cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # Original
        axes[0].imshow(img_pil)
        axes[0].set_title('Original Drawing', fontsize=12, fontweight='bold')
        axes[0].axis('off')
        
        # GradCAM
        axes[1].imshow(cam_image)
        axes[1].set_title('GradCAM Attention', fontsize=12, fontweight='bold')
        axes[1].axis('off')
        
        # Probability distribution
        colors = ['#FF6B6B', '#FF8E72', '#FFC94D', '#95E06C', '#4ECDC4', '#45B7D1']
        bars = axes[2].barh(
            [SHULMAN_NAMES[i] for i in range(NUM_CLASSES)],
            [probs[i] for i in range(NUM_CLASSES)],
            color=colors[:NUM_CLASSES]
        )
        axes[2].set_xlim(0, 1)
        axes[2].set_title('Class Probabilities', fontsize=12, fontweight='bold')
        for bar, p in zip(bars, probs):
            if p > 0.05:
                axes[2].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                           f'{p:.1%}', va='center', fontsize=9)
        
        plt.suptitle(
            f"Prediction: Shulman {pred_score} — {clinical_label} | AD Risk: {risk_level}",
            fontsize=14, fontweight='bold',
            color='green' if pred_score >= 4 else 'red'
        )
        plt.tight_layout()
        plt.show()
    
    return {
        'shulman_score': pred_score,
        'clinical_label': clinical_label,
        'risk_level': risk_level,
        'confidence': float(probs[pred_score]),
        'ad_risk': risk_val,
        'probabilities': {SHULMAN_NAMES[i]: float(probs[i]) for i in range(NUM_CLASSES)}
    }


# ═══════════════════════════════════════════════════════════
# Try it! Upload your own clock drawing
# ═══════════════════════════════════════════════════════════
try:
    from google.colab import files as colab_files
    print("📤 Upload a clock drawing image to test:")
    uploaded = colab_files.upload()
    for filename in uploaded:
        print(f"\n{'='*60}")
        print(f"🔍 Analyzing: {filename}")
        print(f"{'='*60}")
        result = predict_and_visualize(filename)
        print(f"\n📊 Result:")
        for k, v in result.items():
            if k != 'probabilities':
                print(f"   {k}: {v}")
except ImportError:
    # Not on Colab — use a test image from dataset
    print("🖥️  Running locally — using a random test image")
    sample = df_test.sample(1).iloc[0]
    print(f"   Image: {sample['path']}")
    print(f"   True score: {sample['shulman_score']}")
    result = predict_and_visualize(sample['path'])
    print(f"\n📊 Result:")
    for k, v in result.items():
        if k != 'probabilities':
            print(f"   {k}: {v}")

---

## ✅ Summary — CDT Model Training Complete!

### What was trained:
| Component | Details |
|-----------|---------|
| **Model** | CDTNet (EfficientNet-B0 + custom head) |
| **Dataset** | DEMENTIA_DETECTION_CDT — 7,365 clock drawing images + DDPM synthetic |
| **Task** | 6-class Shulman scale (0-5) with AD risk score |
| **Augmentation** | 12 transforms (rotation, perspective, blur, jitter — NO h-flip) + DDPM diffusion |
| **Synthetic Data** | DDPM (cosine schedule, 500 steps) — balances rare Shulman scores |
| **Training** | AdamW + Cosine Annealing + Label Smoothing + Early Stopping |

### Data Augmentation Pipeline:
| Stage | Method | Purpose |
|-------|--------|---------|
| **1. DDPM Synthesis** | Conditional Diffusion (64×64 → 224×224) | Generate novel clock drawings for rare scores |
| **2. Traditional Aug** | 12 transforms (rotation, perspective, etc.) | Online per-batch diversity |
| **3. Weighted Sampling** | Inverse class-frequency weights | Balance mini-batches |

### XAI — 5 Explainability Methods:
| Method | Library | Purpose |
|--------|---------|---------|
| **GradCAM** | pytorch-grad-cam | Fast class-discriminative heatmap |
| **GradCAM++** | pytorch-grad-cam | Better multi-region localization |
| **LIME** | lime | Superpixel-based local explanations |
| **Integrated Gradients** | captum | Pixel-level attribution via path integral |
| **Occlusion Sensitivity** | captum | Systematic region importance mapping |

### Exported files:
| File | Destination | Purpose |
|------|------------|---------|
| `cognitive_cdt_model.pt` | `app/ml/models/` | Backend inference |
| `cognitive_cdt_model_mobile.pt` | — | Mobile/edge deployment |
| `cdt_class_config.json` | `app/ml/config/` | Class names + thresholds |
| `ddpm_cdt_best.pt` | — | Trained diffusion model |

### How it fits in NeuroVerse:
```
Flutter App: User draws clock on Canvas
    ↓ PNG image
FastAPI Backend: CDTPredictor.predict(image)
    ↓ {shulman_score: 3, ad_risk: 0.35, clinical_label: "Mild Impairment"}
fusion_service.py: Combines CDT + TMT + Speech results
    ↓ Final risk assessment
xai_service.py: Multi-XAI (GradCAM + GradCAM++ + LIME + IG + Occlusion)
    ↓ Visual explanations for doctor dashboard
```

### References:
1. Ho et al. (2020) — *"Denoising Diffusion Probabilistic Models"* (NeurIPS)
2. Nichol & Dhariwal (2021) — *"Improved DDPM"* (ICML)
3. Selvaraju et al. (2017) — GradCAM
4. Chattopadhay et al. (2018) — GradCAM++
5. Ribeiro et al. (2016) — LIME
6. Sundararajan et al. (2017) — Integrated Gradients

### NeuroVerse Detection Modules:
- [x] **Speech** — DementiaBank + EWA-DB ✅
- [x] **Clock Drawing (CDT)** — Roboflow Images + DDPM + 5 XAI ✅ (THIS NOTEBOOK)
- [x] **Motor/Spiral** — YOLODatasetFull + HandPD + UCI + DDPM ✅
- [x] **Motor/Meander** — HandPD + DDPM ✅
- [ ] **Gait** — Walking + Balance + Turn-in-Place
- [ ] **TMT** — Trail Making Test
- [ ] **Fusion** — Combine all module scores